# Tone Response And SNR

This notebook asks what happens to a fixed-amplitude voltage
tone at different positions within one coarse channel.

For each fine-channel position we run two direct voltage
spectrograms:

- signal only, to measure coherent tone power at the target bin
- noise only, to estimate the local power-domain noise std

This separates deterministic transfer through the PFB from
random noise scatter.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pfb_response_tools import (
    PFBExperimentConfig,
    detected_intensity_sweep,
    ideal_response,
    local_noise_stats,
    modeled_bandpass_excess_sweep,
    modeled_bandpass_summary,
    noise_overlay_summary,
    normalize_column,
    run_spectrogram,
    tone_response_sweep,
)

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


In [ ]:
config = PFBExperimentConfig()
bins = [8, 32, 64, 96, 128, 160, 192, 224, 248]
rows = tone_response_sweep(config, bins, tone_level=0.005)
rows


In [ ]:
fine_bins = np.asarray([row["fine_bin"] for row in rows])
fine_offset = (fine_bins - config.fftlength / 2) * config.df / 1e3

signal_rel = normalize_column(rows, "signal_power")
noise_rel = normalize_column(rows, "local_noise_std")
snr_rel = normalize_column(rows, "path_snr")
ideal_rel = normalize_column(rows, "ideal_response")

fig, ax = plt.subplots()
ax.plot(fine_offset, ideal_rel, "o-", label="Ideal PFB response")
ax.plot(fine_offset, signal_rel, "o-", label="Signal-only tone power")
ax.plot(fine_offset, noise_rel, "o-", label="Local noise std")
ax.plot(fine_offset, snr_rel, "o-", label="Path-summed SNR")
ax.axhline(1, color="0.25", lw=1, alpha=0.5)
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("Relative to sweep mean")
ax.set_title("Fixed voltage tone across PFB response")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
compact = [
    {
        "fine_bin": row["fine_bin"],
        "ideal_rel": ideal_rel[i],
        "signal_power_rel": signal_rel[i],
        "noise_std_rel": noise_rel[i],
        "path_snr_rel": snr_rel[i],
    }
    for i, row in enumerate(rows)
]
compact


The central bins show the expected behavior: signal power and
local noise both inherit the PFB response, so local-noise SNR is
much less response-dependent than a global-background estimate.

The near-edge bins are different. The coherent tone peak drops
sharply near coarse-channel edges, and peak-bin SNR drops with
it. This is a warning that a single "noise is lower, so SNR is
proportionally higher" rule is not right for voltage-domain
signals. The PFB transfer function acts on the signal too, and
edge behavior likely needs adjacent-coarse-channel accounting
before we use this as a calibrated correction.


For frame-level synthetic injection, this means there are two
separate contracts:

- voltage-domain injection should be calibrated against the PFB
  transfer of both signal and noise
- spectrogram-domain injection with fixed additive intensity
  should optionally adjust intensity by a response/noise model
  if the target is constant local SNR across the band
